# Tracking Config Check — Ants

Validate tracking configs (v1-v5): color bounds, spatial params, detection quality.

**Sections:**
1. **Color palette** — admitted HSV ranges per version.
2. **Detection** — 3-frame diagnostic per version with blue-first matching.
3. **Config summary** — side-by-side comparison across versions.

> To re-optimize color bounds, run `bash scripts/03_tracking/optimize_bounds.sh` then re-run this notebook.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from notebooks.tracking.utils import (
    get_background,
    get_bounds_from_config,
    list_obs_ids,
    load_config,
    load_frame,
    plot_color_palette,
    plot_frame_analysis,
    plot_scale_overview,
)

VERSIONS = ["v1", "v2", "v3", "v4", "v5"]
%matplotlib inline
plt.rcParams["figure.dpi"] = 110

In [ ]:
FRAME_INDICES = [50, 500, 1000, 1500, 2000]

configs = {v: load_config(v) for v in VERSIONS}
bounds  = {v: get_bounds_from_config(v) for v in VERSIONS}

for v in VERSIONS:
    print(f"{v}:  blue {bounds[v]['blue_lower']} → {bounds[v]['blue_upper']}"
          f"   yellow {bounds[v]['yellow_lower']} → {bounds[v]['yellow_upper']}")

---
## 1  Color Palette

Admitted HSV ranges per version. Each panel shows H (x) x S (y) at the config's V_min or V_max.
Bright = accepted region (within H+S bounds). Dimmed = rejected.

In [ ]:
fig = plot_color_palette(configs)
plt.show()

---
## 2  Detection

Scale reference + one sample frame per version, then `FRAME_INDICES` diagnostic rows per version.

| Column | Content |
|--------|---------|
| original | Raw frame |
| no background | Frame minus background |
| ants | Moving-object mask (99th pct threshold); dot per body colored by count (green=1, orange=2, red=3+) |
| blue marking | All blue-matching pixels on full frame (dim) + detected blob (bright), with pixel counts and detection ratio |
| yellow marking | All yellow-matching pixels on full frame (dim) + detected blob (bright), with pixel counts and detection ratio |
| tracking | Identity dots + X at detected color mark. **Red warning** if blue or yellow not identified |

**Color filtering** (HSV on ORIGINAL frame, not bg-subtracted, bounds from per-version config):
- Blue = unique hue — reliable, no overlap with body.
- Yellow = tight S+V filter — ants are black (S~25-35), so high S rejects body.
- At most **one dot per color per frame** → single best blob by saturation (min mean S=70).
- **Detection ratio** = detected blob pixels / total matching pixels. N/A if no matching pixels at all.
- **POV centered on color mark** (more precise than centroid for merged blobs).

In [ ]:
# Scale reference + one sample frame per version
sample_frames = {v: load_frame(v, list_obs_ids(v)[0], frame_idx=FRAME_INDICES[0]) for v in VERSIONS}
fig = plot_scale_overview(sample_frames, configs)
plt.show()

# Per-version detection diagnostic
for v in VERSIONS:
    obs_id = list_obs_ids(v)[0]
    bg = get_background(v, obs_id)
    frames = [load_frame(v, obs_id, frame_idx=fi) for fi in FRAME_INDICES]

    fig = plot_frame_analysis(
        frames, bg, configs[v],
        title=f"{v} / {obs_id}",
        frame_labels=[f"frame {fi}" for fi in FRAME_INDICES],
    )
    plt.show()

---
## 3  Config Summary

In [ ]:
PARAM_KEYS = [
    "quantile", "n_background_frames",
    "max_dist_activity", "max_step",
    "min_area_color", "min_area_body", "max_area_body",
    "pov_radius",
]
BOUND_KEYS = ["blue_lower", "blue_upper", "yellow_lower", "yellow_upper"]

rows_params = {k: {v: configs[v].get(k) for v in VERSIONS} for k in PARAM_KEYS}
rows_bounds = {
    k: {v: list(int(x) for x in bounds[v][k]) for v in VERSIONS}
    for k in BOUND_KEYS
}

df = pd.DataFrame({**rows_params, **rows_bounds}).T
df.index.name = "param"

# Highlight rows where v2–v5 differ
def highlight_diff(row):
    vals_v2_v5 = [row.get(v) for v in ["v2","v3","v4","v5"] if v in row]
    if len(set(str(x) for x in vals_v2_v5)) > 1:
        return ["background-color: #ffe6e6"] * len(row)
    return [""] * len(row)

display(df.style.apply(highlight_diff, axis=1))